In [6]:
import numpy as np
from sympy import *

import torch
from torch import nn
# import torch.nn.functional as F
import torch.optim as optim
# import torch.nn.init as init

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

In [2]:
# t, delta, omega = symbols("t delta omega")
t = Symbol("t", positive=True)

m = 1
mu = 5
k = 500

delta = mu/(2*m)
w0 = sqrt(k)
w = sqrt(w0**2 - delta**2)

phi = tan(-delta/w)**(-1)
A = 1/(2 * cos(phi))

display(A)

x = exp(-delta*t)*(2*A*cos(phi + w*t))
display(x.simplify())

-0.595470930409999

-1.19094186082*exp(-2.5*t)*cos(22.220486043289*t - 8.85065980104099)

In [13]:
target_train = np.random.rand(10) * 0.5
x_train = np.array([x.subs(t, time) for time in target_train])
print(target_train)
print(x_train)


# create validation data
target_val = np.random.rand(10) * 0.5
x_val = np.array([x.subs(t, time) for time in target_val])
print(target_val)
print(x_val)

[0.15610176 0.18727293 0.28576939 0.40341707 0.46155804 0.08091815
 0.04886664 0.36121011 0.40343124 0.14765272]
[-0.500351489830167 0.0171679260899479 0.467267746764243
 -0.431603022854064 -0.0618508153239818 -0.698784852710175
 -0.0938513249113661 -0.327779653022853 -0.431572213155972
 -0.622538672574275]
[0.27865498 0.15541213 0.31719015 0.08838184 0.24394525 0.48628344
 0.49400604 0.10379015 0.12006239 0.37642728]
[0.525575076619916 -0.510857954664195 0.123771352593637 -0.786125573452769
 0.620443463053552 0.132287283334666 0.182689255135489 -0.887594238262988
 -0.877694155770055 -0.410850796129008]


In [14]:
simple_model = nn.Sequential(
    # input layer with 1 input
    nn.Linear(1, 10),
    nn.Tanh(),
    # hidden layer
    nn.Linear(10, 1)
)

print(simple_model)
net = simple_model

Sequential(
  (0): Linear(in_features=1, out_features=10, bias=True)
  (1): Tanh()
  (2): Linear(in_features=10, out_features=1, bias=True)
)


In [15]:
optimizer = optim.SGD(net.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [ ]:
# we could have done this ourselves,
# but we should be aware of sklearn and its tools
from sklearn.metrics import accuracy_score

# setting hyperparameters and gettings epoch sizes
batch_size = 1
num_epochs = 100
num_samples_train = len(x_train)
num_batches_train = num_samples_train // batch_size
num_samples_valid = len(target_val)
num_batches_valid = num_samples_valid // batch_size

# setting up lists for handling loss/accuracy
train_acc, train_loss = [], []
valid_acc, valid_loss = [], []
test_acc, test_loss = [], []
cur_loss = 0
losses = []

get_slice = lambda i, size: range(i * size, (i + 1) * size)

for epoch in range(num_epochs):
    # Forward -> Backprob -> Update params
    ## Train
    cur_loss = 0
    net.train()
    for i in range(num_batches_train):
        optimizer.zero_grad()
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])
        
        # compute gradients given loss
        target_batch = target_train[slce]
        batch_loss = criterion(output, target_batch)
        batch_loss.backward()
        optimizer.step()
        
        cur_loss += batch_loss   
    losses.append(cur_loss / batch_size)

    net.eval()
    ### Evaluate training
    train_preds, train_targs = [], []
    for i in range(num_batches_train):
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])
        
        preds = torch.max(output, 1)[1]
        
        train_targs += list(target_train[slce].numpy())
        train_preds += list(preds.data.numpy())
    
    ### Evaluate validation
    val_preds, val_targs = [], []
    for i in range(num_batches_valid):
        slce = get_slice(i, batch_size)